In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, classification_report,
confusion_matrix, roc_auc_score, roc_curve)
import joblib
import warnings
warnings.filterwarnings('ignore')


In [2]:
X = pd.read_csv("../data/X_ml_ready.csv")
y = pd.read_csv("../data/y_ml_ready.csv").values.ravel()

In [3]:
print(f"Data loaded: X={X.shape}, y={len(y)}")
print("Target distribution:")
print(pd.Series(y).value_counts(normalize=True).round(2))


Data loaded: X=(50000, 33), y=50000
Target distribution:
0    0.7
1    0.3
Name: proportion, dtype: float64


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
X, y, test_size=0.2, random_state=42, stratify=y)
print(f"   Training: {X_train.shape[0]:,} samples")
print(f"   Testing:  {X_test.shape[0]:,} samples")

   Training: 40,000 samples
   Testing:  10,000 samples


In [11]:
# LOGISTIC REGRESSION
log_model = LogisticRegression(max_iter=2000, random_state=42)
log_model.fit(X_train, y_train)
y_pred_log = log_model.predict(X_test)
log_acc = accuracy_score(y_test, y_pred_log)
log_auc = roc_auc_score(y_test, log_model.predict_proba(X_test)[:,1])

In [12]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_split=20,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)
rf_acc = accuracy_score(y_test, y_pred_rf)
rf_auc = roc_auc_score(y_test, rf_model.predict_proba(X_test)[:,1])

In [13]:
rf_cv_scores = cross_val_score(rf_model, X_train, y_train, cv=5, scoring='accuracy')
log_cv_scores = cross_val_score(log_model, X_train, y_train, cv=5, scoring='accuracy')

print(f"   RF CV Accuracy: {rf_cv_scores.mean():.3f} ± {rf_cv_scores.std():.3f}")
print(f"   Logistic CV:     {log_cv_scores.mean():.3f} ± {log_cv_scores.std():.3f}")

   RF CV Accuracy: 0.887 ± 0.001
   Logistic CV:     0.879 ± 0.001


In [14]:
models = {
    'Logistic Regression': {'model': log_model, 'acc': log_acc, 'auc': log_auc, 'cv': log_cv_scores.mean()},
    'Random Forest': {'model': rf_model, 'acc': rf_acc, 'auc': rf_auc, 'cv': rf_cv_scores.mean()}
}

best_model_name = max(models.keys(), key=lambda k: models[k]['acc'])
best_model = models[best_model_name]['model']
best_acc = models[best_model_name]['acc']

print(f"\n Best model Seleted: {best_model_name}")
print(f"Test Accuracy: {best_acc:.3f}")
print(f"Cross-Val Stable: {models[best_model_name]['cv']:.3f}")


 Best model Seleted: Random Forest
Test Accuracy: 0.888
Cross-Val Stable: 0.887


In [15]:
import joblib
import os
os.makedirs("../models", exist_ok=True)
joblib.dump(rf_model, "../models/production_model.pkl")



['../models/production_model.pkl']

In [16]:
print("\n TOP FEATURES:")
importance = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False)
print(importance.head(10))


 TOP FEATURES:
technical_score                 0.265685
years_of_experience             0.184685
interview_score                 0.114536
skills_match_percentage         0.071583
job_role_match_Not Matched      0.066014
expected_ctc_lpa                0.057443
internship_experience_Yes       0.031612
previous_ctc_lpa                0.027352
relevant_experience_Relevant    0.026720
skills_match_level_Low          0.023555
dtype: float64
